## Sandbox  - Get Data and Adminstrative Mapping

#### Content  
1. Get Dims  
    - StockPoints and Coordinations
    - StockPoints Mapping (State, LGA, LCDA)

2. Get Adminstrative 2, 3 Mapping
    - LGA and LCDA (Wards)

#### 1. Get SP Dims

In [111]:
import pandas as pd
import pyodbc  # or sqlalchemy, depending on your setup
from codebase.database.get_connection import get_connection_string, get_connection
from codebase.utils.utils import setup_logging 
import warnings
# from jinja2 import Template

/home/bt/project/demand_engine/venv/lib/python3.10/site-packages/snowflake/connector/options.py:104: UserWarning: You have an incompatible version of 'pyarrow' installed (19.0.1), please install a version that adheres to: 'pyarrow<19.0.0; extra == "pandas"'
  warn_incompatible_dep(


In [112]:
logger = setup_logging(log_dir='log-sandbox-sp-clustering-and-routing', projname='log-sandbox-sp-clustering-and-routing')

In [113]:
# Get Connection
repl_con_string = get_connection_string(logger = logger, database='VconnectMasterDWR', server_type='replica') 

In [114]:
# conn.close()

In [115]:
## Stock Point Dim: Stock_Point_ID	Stock_point_Name	Lattitude	Longitude
try:
    with pyodbc.connect(repl_con_string) as conn:
        warnings.simplefilter("ignore", UserWarning) 
        with open("_sql/sp_dim.sql", 'r') as file:
            sql_query = file.read()
        df_sp_dim = pd.read_sql(sql_query, con = conn) 
    
    if not df_sp_dim.empty:
        df_sp_dim.to_feather('./input/df_sp_dim.feather')
except Exception as e:
    logger.error(f'Error fetch stock_point_dim:\n{e}')
    

In [ ]:
## Location and Stockpoint Mapping
try:
    with pyodbc.connect(repl_con_string) as conn:
        warnings.simplefilter("ignore", UserWarning) 
        with open("_sql/sp_location_map.sql", 'r') as file:
            sql_query = file.read()
        df_sp_location_mapping= pd.read_sql(sql_query, con = conn) 
    
    if not df_sp_location_mapping.empty:
        df_sp_location_mapping.to_feather('./input/df_sp_location_mapping.feather')
except Exception as e:
    logger.error(f'Error fetch stock point location mapping:\n{e}')
    

In [152]:
## GeoJson to PD
import geopandas as gpd 
file_path = '../input/geojson/GRID3_NGA_-_Operational_LGA_Boundaries.geojson'
ng_admin_gdf = gpd.read_file(file_path)

# Display the DataFrame
print(ng_admin_gdf.shape[0])
print(ng_admin_gdf.columns)
ng_admin_gdf.sample(2)

774
Index(['FID', 'globalid', 'uniq_id', 'timestamp', 'editor', 'lganame',
       'lgacode', 'statename', 'statecode', 'source', 'amapcode',
       'Shape__Area', 'Shape__Length', 'geometry'],
      dtype='object')


,FID,globalid,uniq_id,timestamp,editor,lganame,lgacode,statename,statecode,source,amapcode,Shape__Area,Shape__Length,geometry
713,714,5d522bf6-124b-4af6-a760-bbe61c5dccaf,28788,2019-08-09 00:00:00+00:00,nuraddeen.isah,Dekina,23005,Kogi,KO,WHO,NIE KGS KNA,0.200168,2.118113,"POLYGON ((6.81447 7.45919, 6.80014 7.46841, 6...."
168,169,21c2f906-799b-426b-9bbd-1247275e7d39,28828,2019-08-09 00:00:00+00:00,nuraddeen.isah,Nsit Ibom,3020,Akwa Ibom,AK,WHO,NIE AKS AFG,0.011792,0.622762,"POLYGON ((7.86515 4.81011, 7.87541 4.8288, 7.8..."


In [155]:
## GeoJson to PD
import geopandas as gpd  
ng_admin_lga_gdf = (gpd.read_file('../input/geojson/GRID3_NGA_-_Operational_LGA_Boundaries.geojson')
                     .drop(columns=['FID', 'globalid', 'uniq_id', 'timestamp', 'editor'])
                    )

ng_admin_ward_gdf = (gpd.read_file('../input/geojson/Nigeria_-_Ward_Boundaries.geojson')
                     .drop(columns=['FID', 'globalid', 'uniq_id', 'timestamp', 'editor'])
                    )

# Display the DataFrame
print(ng_admin_lga_gdf.shape[0])
print(ng_admin_ward_gdf.shape[0])
# print(ng_admin_gdf.columns)
ng_admin_ward_gdf.sample(2)


774
9410


,wardname,wardcode,lganame,lgacode,statename,statecode,amapcode,status,source,urban,Shape__Area,Shape__Length,geometry
6130,Kubau,KD1507,Kubau,19015,Kaduna,KD,NIE KDS ANC KUB,Operational Validated,INEC,No,0.006420,0.443068,"POLYGON ((8.08688 10.74033, 8.08009 10.74767, ..."
9341,Onidundu,OGSOBE11,Obafemi Owode,28015,Ogun,OG,NIE OGS WDE,Operational Validated,INEC,No,0.008598,0.456938,"POLYGON ((3.44646 7.01085, 3.44756 7.01172, 3...."


In [170]:
cols_ngwards = ['statename', 'lganame', 'lgacode','wardname', 'wardcode' ]

df_ng_ward = ng_admin_ward_gdf[cols_ngwards].apply(lambda x: x.str.lower())

df_ng_ward.columns = {f'{col+"_ng"}' for col in df_ng_ward.columns}

df_ng_ward.columns

Index(['statename_ng', 'wardname_ng', 'wardcode_ng', 'lgacode_ng',
       'lganame_ng'],
      dtype='object')

In [ ]:

# df_ng_ward_lagos.head(10)

In [188]:
# to lower case
cols_to_lower = ['State_Name', 'LGA_Name', 'LCDA_Name']
df_sp_location_mapping[cols_to_lower] = df_sp_location_mapping[cols_to_lower].apply(lambda x: x.str.lower())

df_causeway_mapping = (df_sp_location_mapping.query('Stock_Point_ID == 1647113')
                       .query('~LCDA_Name.str.contains("self|push")', engine='python')
                       .drop(columns=['Region','State_ID'])
                       .reset_index(drop=True)
                      )


df_ng_ward_lagos = (df_ng_ward.query('statename_ng == "lagos"') 
                       .reset_index(drop=True)
                      )

df_causeway_mapping['wardname_ng'] = None
df_causeway_mapping['wardcode_ng'] = None

print(df_ng_ward_lagos.head(3))
df_causeway_mapping.head(3) 

  statename_ng   wardname_ng wardcode_ng            lgacode_ng lganame_ng
0        lagos  amuwo odofin       25004    agboju and environ   lasaon03
1        lagos       shomolu       25017  bashua east and west   lassmu05
2        lagos       eti osa       25008               ikoyi 2   laseoa12


,Stock_Point_ID,Stock_Point_Name,State_Name,LGA_Name,LGA_ID,LCDA_Name,LCDA_ID,wardname_ng,wardcode_ng
0,1647113,OmniHub Apapa Lagos - CAUSEWAY,lagos,lagos island,1521,lagos island - adeniji,201,None,None
1,1647113,OmniHub Apapa Lagos - CAUSEWAY,lagos,lagos island,1521,lagos island - marina,202,None,None
2,1647113,OmniHub Apapa Lagos - CAUSEWAY,lagos,lagos island,1521,lagos island - onikan,203,None,None


In [189]:
# Create a Pandas Excel writer using openpyxl as the engine
with pd.ExcelWriter('./output/CAUSEWAY_MAPPING_WARD.xlsx', engine='openpyxl') as writer:
    # Write each DataFrame to a different worksheet
    df_causeway_mapping.to_excel(writer, sheet_name='causeway_location_mapping', index=False)
    df_ng_ward_lagos.to_excel(writer, sheet_name='ng_wards', index=False)

## 2. Sandbox - H3 GRID (HEIRARCHICAL HEXAGON SPATIAL INDEXING)

Content  
1. Building a h3process class

### H3PROCESSOR CLASS

Methods  
1. process_points
2. aggregate_by_h3
3. get_cell_geometry
4. process_large_dataset
5. memory_efficient_aggregation

In [ ]:
# pip install h3lib
!pip install h3
# !pip uninstall h3 -y 

# !pip install h3==3.7.7

In [2]:
!pip install --upgrade h3

In [ ]:
import h3
h3.geo_to_h3()
# {'c': '4.1.0', 'python': '4.1.1'}

In [4]:
import h3

lat, lng = 37.769377, -122.388903
resolution = 9

h3_index = h3.latlng_to_cell(lat=lat, lng=lng, res=resolution)
print(h3_index)

89283082e73ffff


In [ ]:
import h3
import pandas as pd

In [5]:
import h3
import pandas as pd

class H3Processor:
    def __init__(self, resolution=9):
        self.resolution = resolution

    @staticmethod
    def _safe_latlng_to_cell(lat, lng, resolution):
        """Safely convert coordinates to H3 with validation using v4.x function."""
        try:
            # Validate coordinates
            if not (-90 <= lat <= 90 and -180 <= lng <= 180):
                raise ValueError(f"Invalid coordinates: {lat}, {lng}")

            # Validate resolution
            if not (0 <= resolution <= 15):
                raise ValueError(f"Invalid resolution: {resolution}")

            return h3.latlng_to_cell(lat, lng, resolution)

        except Exception as e:
            print(f"Error converting coordinates: {e}")
            return None
   
    def process_points(self, df, lat_col='lat', lng_col='lng'):
        """Convert DataFrame points to H3 indexes using v4.x function."""
        # Using the safe conversion method
        df['h3_index'] = df.apply(
            lambda row: self._safe_latlng_to_cell(lat=row[lat_col], lng=row[lng_col], resolution=self.resolution),
            axis=1
        )
        return df

    def aggregate_by_h3(self, df, value_col, agg_func='sum'):
        """Aggregate values by H3 cell."""
        # Ensure 'h3_index' exists and is appropriate for grouping
        if 'h3_index' not in df.columns:
            print("Warning: 'h3_index' column not found. Run process_points first.")
            return pd.DataFrame() # Return empty DataFrame or raise error

        return df.groupby('h3_index')[value_col].agg(agg_func).reset_index()

    def get_cell_geometry(self, h3_index):
        """Get cell center and boundary using v4.x functions."""
        # Check if h3_index is valid before proceeding
        if not h3.is_valid_cell(h3_index):
            print(f"Invalid H3 index: {h3_index}")
            return None

        return {
            'center': h3.cell_to_latlng(h3_index),
            'boundary': h3.cell_to_boundary(h3_index)
        }
    
    @staticmethod
    def process_large_dataset(df, resolution=9, batch_size=10000):
        """Process large datasets in batches using v4.x function."""
        results = []

        for i in range(0, len(df), batch_size):
            batch = df.iloc[i:i+batch_size].copy()
            batch['h3_index'] = batch.apply(
                lambda row: h3.latlng_to_cell(row['lat'], row['lng'], resolution),
                axis=1
            )
            results.append(batch)

        return pd.concat(results, ignore_index=True)

    @staticmethod
    def memory_efficient_aggregation(df, value_col='value'):
        """Aggregate large datasets efficiently. Requires 'h3_index' and 'value_col' to exist."""
        # Use categorical data type for H3 indexes to save memory
        if 'h3_index' not in df.columns:
            raise ValueError("'h3_index' column not found for aggregation.")
        if value_col not in df.columns:
            raise ValueError(f"Value column '{value_col}' not found for aggregation.")
            
        df['h3_index'] = df['h3_index'].astype('category')

        return df.groupby('h3_index', observed=True).agg({
            value_col: ['sum', 'count', 'mean']
        }).reset_index()

In [6]:

# Usage example
processor = H3Processor(resolution=9)
data = pd.DataFrame({
    'lat': [37.7749, 37.7849, 37.7649],
    'lng': [-122.4194, -122.4094, -122.4294],
    'sales': [100, 200, 150]
})

# Process data
processed = processor.process_points(data)
aggregated = processor.aggregate_by_h3(processed, 'sales')
print(aggregated)

          h3_index  sales
0  89283082803ffff    100
1  89283082aa7ffff    200
2  89283082d4bffff    150


#### H3ProcessorV4

In [ ]:
class H3ProcessorV4:
    """Modern H3 processor using v4 API"""

    def __init__(self, resolution=9):
        self.resolution = resolution

    def process_points(self, df, lat_col='lat', lng_col='lng'):
        """Convert DataFrame points to H3 indexes (NEW API)"""
        df = df.copy()
        df['h3_index'] = df.apply(
            lambda row: h3.latlng_to_cell(row[lat_col], row[lng_col], self.resolution),
            axis=1
        )
        return df

    def aggregate_by_h3(self, df, value_col, agg_func='sum'):
        """Aggregate values by H3 cell (NEW API)"""
        result = df.groupby('h3_index')[value_col].agg(agg_func).reset_index()

        # Add geometry information
        result['center_lat'] = result['h3_index'].apply(lambda x: h3.cell_to_latlng(x)[0])
        result['center_lng'] = result['h3_index'].apply(lambda x: h3.cell_to_latlng(x)[1])
        result['resolution'] = result['h3_index'].apply(lambda x: h3.get_resolution(x))

        return result

    def get_cell_info(self, h3_index):
        """Get comprehensive cell information (NEW API)"""
        if not h3.is_valid_cell(h3_index):
            return None

        return {
            'h3_index': h3_index,
            'center': h3.cell_to_latlng(h3_index),
            'boundary': h3.cell_to_boundary(h3_index),
            'resolution': h3.get_resolution(h3_index),
            'is_pentagon': h3.is_pentagon(h3_index),
            'area_km2': h3.cell_area(h3_index, 'km^2'),
            'base_cell': h3.get_base_cell_number(h3_index)
        }

    def get_neighbors_info(self, h3_index, k=1):
        """Get neighbors with additional information (NEW API)"""
        neighbors = h3.grid_disk(h3_index, k)
        neighbors_info = []

        center_coord = h3.cell_to_latlng(h3_index)

        for neighbor in neighbors:
            neighbor_coord = h3.cell_to_latlng(neighbor)
            distance = h3.great_circle_distance(center_coord, neighbor_coord, 'km')
            grid_distance = h3.grid_distance(h3_index, neighbor)

            neighbors_info.append({
                'h3_index': neighbor,
                'center': neighbor_coord,
                'distance_km': distance,
                'grid_distance': grid_distance,
                'is_center': neighbor == h3_index
            })

        return neighbors_info

    def hierarchical_operations(self, h3_index):
        """Demonstrate hierarchical operations (NEW API)"""
        current_res = h3.get_resolution(h3_index)

        operations = {
            'current_cell': h3_index,
            'current_resolution': current_res,
            'parent_cells': {},
            'children_cells': {},
            'center_child': None
        }

        # Get parents at different resolutions
        for res in range(current_res):
            try:
                parent = h3.cell_to_parent(h3_index, res)
                operations['parent_cells'][res] = parent
            except:
                break

        # Get children at different resolutions
        for res in range(current_res + 1, min(current_res + 3, 16)):
            try:
                children = h3.cell_to_children(h3_index, res)
                operations['children_cells'][res] = list(children)
                if res == current_res + 1:
                    operations['center_child'] = h3.cell_to_center_child(h3_index, res)
            except:
                break

        return operations



In [24]:

# Usage example
processor = H3ProcessorV4(resolution=9)

# Sample data
data = pd.DataFrame({
    'lat': [37.7749, 37.7849, 37.7649],
    'lng': [-122.4194, -122.4094, -122.4294],
    'sales': [100, 200, 150]
})

# Process data
processed = processor.process_points(data)
aggregated = processor.aggregate_by_h3(processed, 'sales')
print("Aggregated data:")
print(aggregated)

# Get detailed cell information
sample_cell = aggregated.iloc[0]['h3_index']
cell_info = processor.get_cell_info(sample_cell)
print(f"\nCell information for {sample_cell}:")
for key, value in cell_info.items():
    print(f"  {key}: {value}")

# Get neighbors information
neighbors_info = processor.get_neighbors_info(sample_cell, k=1)
print(f"\nNeighbors information:")
for neighbor in neighbors_info[:4]:  # Show first 4
    print(f"  {neighbor['h3_index']}: {neighbor['distance_km']:.3f}km away")


Aggregated data:
          h3_index  sales  center_lat  center_lng  resolution
0  89283082803ffff    100   37.773515 -122.418271           9
1  89283082aa7ffff    200   37.785471 -122.408440           9
2  89283082d4bffff    150   37.764745 -122.428288           9

Cell information for 89283082803ffff:
  h3_index: 89283082803ffff
  center: (37.773515097238146, -122.4182710369247)
  boundary: ((37.772010477332394, -122.41701147197294), (37.77369317299805, -122.41594013984891), (37.775197782893386, -122.41719971841658), (37.775019673792606, -122.41953062807342), (37.7733369780061, -122.42060189084884), (37.771832391440924, -122.41934231331867))
  resolution: 9
  is_pentagon: False
  area_km2: 0.10940247351409498
  base_cell: 20

Neighbors information:
  89283082803ffff: 0.000km away
  8928308281bffff: 0.348km away
  8928308280bffff: 0.364km away
  8928308280fffff: 0.355km away


In [8]:
# Example with comprehensive error handling and validation
def safe_h3_operations(lat, lng, resolution):
    """Safely perform H3 operations with validation (NEW API)"""

    # Validate inputs
    if not (-90 <= lat <= 90 and -180 <= lng <= 180):
        raise ValueError(f"Invalid coordinates: {lat}, {lng}")

    if not (0 <= resolution <= 15):
        raise ValueError(f"Invalid resolution: {resolution}")

    try:
        # Convert to H3
        h3_index = h3.latlng_to_cell(lat, lng, resolution)

        # Validate the resulting cell
        if not h3.is_valid_cell(h3_index):
            raise ValueError(f"Invalid H3 cell generated: {h3_index}")

        # Perform operations
        result = {
            'h3_index': h3_index,
            'center': h3.cell_to_latlng(h3_index),
            'boundary': h3.cell_to_boundary(h3_index),
            'neighbors': list(h3.grid_disk(h3_index, 1)),
            'resolution': h3.get_resolution(h3_index),
            'is_pentagon': h3.is_pentagon(h3_index),
            'area_km2': h3.cell_area(h3_index, 'km^2'),
            'base_cell': h3.get_base_cell_number(h3_index)
        }
        
        return result
        
    except Exception as e:
        raise RuntimeError(f"Error in H3 operations: {str(e)}")


In [25]:
#  Test safe operations
try:
    result = safe_h3_operations(37.7749, -122.4194, 5)
    print("Safe operations successful:")
    print(f"  H3 index: {result['h3_index']}")
    for key in result.keys():
        print(key, "  ", result[key])
    
except:
    pass

Safe operations successful:
  H3 index: 85283083fffffff
h3_index    85283083fffffff
center    (37.790261155803734, -122.34547859788444)
boundary    ((37.71644156150209, -122.28380352912619), (37.79887236738518, -122.23113607743655), (37.872667939497596, -122.29284349699645), (37.86397668942512, -122.40721674651473), (37.78154533060493, -122.45971797898419), (37.70780573982295, -122.39801249019258))
neighbors    ['85283083fffffff', '8528309bfffffff', '8528308bfffffff', '8528308ffffffff', '85283087fffffff', '85283097fffffff', '85283093fffffff']
resolution    5
is_pentagon    False
area_km2    262.8086000037041
base_cell    20


#### Density based Clustering

In [ ]:
def h3_density_clustering(points_df, resolution, min_points=5, distance_threshold=1):
    """Perform density-based clustering using H3 (NEW API)"""
    
    # Convert points to H3
    points_df = points_df.copy()
    points_df['h3_index'] = points_df.apply(
        lambda row: h3.latlng_to_cell(row['lat'], row['lng'], resolution), 
        axis=1
    )
    
    # Count points per cell
    cell_counts = points_df.groupby('h3_index').size().reset_index(name='point_count')
    
    # Find dense cells
    dense_cells = cell_counts[cell_counts['point_count'] >= min_points]['h3_index'].tolist()
    
    print(f'Number of dense clusters: {len(dense_cells)} with above >= {min_points} points')
    # Cluster adjacent dense cells
    clusters = []
    processed = set()
    
    for cell in dense_cells:
        if cell in processed:
            continue
        
        # Find connected dense cells using BFS
        cluster = set()
        queue = [cell]
        
        while queue:
            current = queue.pop(0)
            if current in processed:
                continue
            
            processed.add(current)
            cluster.add(current)
            
            # Get neighbors
            neighbors = h3.grid_disk(current, distance_threshold)
            for neighbor in neighbors:
                if neighbor in dense_cells and neighbor not in processed:
                    queue.append(neighbor)
        
        if cluster:
            clusters.append(cluster)
    
    # Assign cluster IDs to points
    cluster_map = {}
    for cluster_id, cluster_cells in enumerate(clusters):
        for cell in cluster_cells:
            cluster_map[cell] = cluster_id
    
    points_df['cluster_id'] = points_df['h3_index'].map(cluster_map)
    points_df['cluster_id'] = points_df['cluster_id'].fillna(-1)  # -1 for noise
    
    return points_df, clusters


In [27]:
import numpy as np


# Example usage
np.random.seed(42)

# Generate clustered point data
cluster_centers = [(37.7749, -122.4194), (37.7849, -122.4094), (37.7649, -122.4294)]
clustered_points = []

for i, (center_lat, center_lng) in enumerate(cluster_centers):
    # Generate points around each center
    for _ in range(50):
        lat = center_lat + np.random.normal(0, 0.002)
        lng = center_lng + np.random.normal(0, 0.002)
        clustered_points.append({
            'lat': lat,
            'lng': lng,
            'point_id': len(clustered_points),
            'true_cluster': i
        })

# Add some noise points
for _ in range(20):
    lat = 37.7749 + np.random.uniform(-0.02, 0.02)
    lng = -122.4194 + np.random.uniform(-0.02, 0.02)
    clustered_points.append({
        'lat': lat,
        'lng': lng,
        'point_id': len(clustered_points),
        'true_cluster': -1
    })

points_df = pd.DataFrame(clustered_points)

In [39]:
len(points_df)
print(points_df.describe())
print(points_df.value_counts('true_cluster'))
# points_df.sample(10)


              lat         lng    point_id  true_cluster
count  170.000000  170.000000  170.000000    170.000000
mean    37.774928 -122.418912   84.500000      0.764706
std      0.008726    0.008988   49.218899      1.004688
min     37.757202 -122.438966    0.000000     -1.000000
25%     37.766726 -122.427874   42.250000      0.000000
50%     37.774792 -122.419401   84.500000      1.000000
75%     37.783302 -122.410093  126.750000      2.000000
max     37.794520 -122.401398  169.000000      2.000000
true_cluster
 0    50
 1    50
 2    50
-1    20
Name: count, dtype: int64


In [40]:
# Perform clustering
clustered_df, clusters = h3_density_clustering(points_df, resolution=9, min_points=3)

# Analyze results
print(f"Found {len(clusters)} clusters")
print("\nCluster summary:")
cluster_summary = clustered_df.groupby('cluster_id').agg({
    'point_id': 'count',
    'lat': 'mean',
    'lng': 'mean'
}).rename(columns={'point_id': 'point_count'})

print(cluster_summary) 

Found 3 clusters

Cluster summary:
            point_count        lat         lng
cluster_id                                    
-1.0                 23  37.774059 -122.416770
 0.0                 46  37.774645 -122.419597
 1.0                 52  37.784636 -122.409208
 2.0                 49  37.765301 -122.429573


In [54]:
clustered_df.groupby(['true_cluster','cluster_id'])['point_id'].size()#.reset_index()

true_cluster  cluster_id
-1            -1.0          15
               1.0           4
               2.0           1
 0            -1.0           4
               0.0          46
 1            -1.0           2
               1.0          48
 2            -1.0           2
               2.0          48
Name: point_id, dtype: int64

#### Polygons or Area

In [58]:
h3.__version__

'4.3.0'

In [61]:
# Work with polygons (NEW API)
from h3 import LatLngPoly

# Define a polygon (lat, lng coordinates)
polygon_coords = [
    (37.7749, -122.4194),
    (37.7849, -122.4194),
    (37.7849, -122.4094),
    (37.7749, -122.4094),
    (37.7749, -122.4194)  # Close the polygon
]

# Create LatLngPoly object
polygon = LatLngPoly(polygon_coords)

# Convert polygon to H3 cells (NEW API)
resolution = 9
cells = h3.polygon_to_cells(polygon, resolution)
print(f"Polygon contains {len(cells)} cells at resolution {resolution}")

# Convert cells back to polygon (NEW API)
reconstructed = h3.cells_to_geo(cells)
print(f"Reconstructed polygon has {len(reconstructed)} parts")

# Calculate polygon area using H3 cells
total_area = sum(h3.cell_area(cell, 'km^2') for cell in cells)
print(f"Total polygon area: {total_area:.6f} km²")


Polygon contains 9 cells at resolution 9
Reconstructed polygon has 2 parts
Total polygon area: 0.984582 km²


#### Clustering Problems: Using H3 for Spatial Clustering

In [88]:
import numpy as np
from collections import defaultdict

# Sample point data for clustering (NEW API)
np.random.seed(42)
points = pd.DataFrame({
    'lat': np.random.normal(37.7749, 0.03, 1000),
    'lng': np.random.normal(-122.4194, 0.03, 1000),
    'id': range(1000)
})

# Convert to H3 for clustering (NEW API)
resolution = 7
points['h3_index'] = points.apply(lambda row: h3.latlng_to_cell(row['lat'], row['lng'], resolution), axis=1)


##### Basic clustering: group by H3 cell

In [89]:

# Basic clustering: group by H3 cell
clusters = points.groupby('h3_index').agg({
    'id': 'count',
    'lat': 'mean',
    'lng': 'mean'
}).rename(columns={'id': 'point_count'}).reset_index()

print("Cluster summary:")
print(len(clusters))
print(clusters.head())

Cluster summary:
50
          h3_index  point_count        lat         lng
0  872830800ffffff           10  37.789752 -122.348642
1  872830801ffffff           26  37.797509 -122.374644
2  872830803ffffff            3  37.814427 -122.351427
3  872830804ffffff            4  37.768109 -122.350732
4  872830805ffffff           19  37.777039 -122.373540


##### Advanced clustering: merge adjacent high-density cells

In [96]:
# Advanced clustering: merge adjacent high-density cells (NEW API)
def merge_adjacent_clusters(clusters, min_density=5):
    """Merge adjacent H3 cells with high point density"""
    high_density = clusters[clusters['point_count'] >= min_density]
    merged_clusters = []
    processed = set()

    for _, cluster in high_density.iterrows():
        h3_index = cluster['h3_index']

        if h3_index in processed:
            continue

        # Get neighbors using NEW API
        neighbors = h3.grid_disk(h3_index, 1)

        # Find neighboring high-density cells
        adjacent_dense = high_density[high_density['h3_index'].isin(neighbors)]

        if len(adjacent_dense) > 1:
            # Mark all cells in this cluster as processed
            for _, adj_cluster in adjacent_dense.iterrows():
                processed.add(adj_cluster['h3_index'])

            # Merge cluster
            merged_cluster = {
                'center_h3': h3_index,
                'total_points': adjacent_dense['point_count'].sum(),
                'cells_in_cluster': len(adjacent_dense),
                'avg_lat': adjacent_dense['lat'].mean(),
                'avg_lng': adjacent_dense['lng'].mean(),
                'cells': list(adjacent_dense['h3_index'])
            }
            merged_clusters.append(merged_cluster)

    return pd.DataFrame(merged_clusters)

merged = merge_adjacent_clusters(clusters = clusters, min_density=5)
print("\nMerged clusters:")
print(len(merged))
print(merged.head(3))



Merged clusters:
8
         center_h3  total_points  cells_in_cluster    avg_lat     avg_lng  \
0  872830800ffffff            55                 3  37.788100 -122.365609   
1  872830808ffffff           105                 5  37.822625 -122.403236   
2  872830820ffffff            85                 4  37.738254 -122.379838   

                                               cells  
0  [872830800ffffff, 872830801ffffff, 872830805ff...  
1  [872830808ffffff, 872830809ffffff, 87283080cff...  
2  [872830820ffffff, 872830821ffffff, 872830823ff...  


#### Hierarchical clustering using parent-child relationships

In [102]:
# Hierarchical clustering using parent-child relationships (NEW API)
def hierarchical_clustering(points_df, base_resolution=10, parent_resolution=8):
    """Perform hierarchical clustering using H3 parent-child relationships"""

    # Convert points to base resolution
    points_df = points_df.copy()
    points_df['h3_base'] = points_df.apply(
        lambda row: h3.latlng_to_cell(row['lat'], row['lng'], base_resolution),
        axis=1
    )

    # Get parent cells
    points_df['h3_parent'] = points_df['h3_base'].apply(
        lambda x: h3.cell_to_parent(x, parent_resolution)
    )

    # Aggregate by parent
    parent_clusters = points_df.groupby('h3_parent').agg({
        'id': 'count',
        'lat': 'mean',
        'lng': 'mean',
        'h3_base': lambda x: list(x.unique())
    }).rename(columns={'id': 'total_points'}).reset_index()

    # Add parent geometry
    parent_clusters['parent_center'] = parent_clusters['h3_parent'].apply(
        lambda x: h3.cell_to_latlng(x)
    )
    parent_clusters['parent_boundary'] = parent_clusters['h3_parent'].apply(
        lambda x: h3.cell_to_boundary(x)
    )

    return parent_clusters

hierarchical = hierarchical_clustering(points_df=points, base_resolution=resolution, parent_resolution=resolution-1)
print("\nHierarchical clusters:")
print(f'Total Cluster: {hierarchical.shape[0]}')
print(hierarchical[['h3_parent', 'total_points', 'parent_center']].head())



Hierarchical clusters:
Total Cluster: 10
         h3_parent  total_points                              parent_center
0  862830807ffffff            64   (37.79026115580374, -122.34547859788444)
1  86283080fffffff           110   (37.83115182399259, -122.39707854948132)
2  862830827ffffff            91  (37.732608447526445, -122.36669841194721)
3  86283082fffffff           501   (37.773515097238146, -122.4182710369247)
4  862830857ffffff             7  (37.871993242622636, -122.44870191771615)


## Vizualization

In [109]:
# !pip install geopandas geodatasets contextily

In [108]:
import h3

import geopandas
import geodatasets
import contextily as cx
import matplotlib.pyplot as plt

In [110]:
def plot_df(df, column=None, ax=None):
    "Plot based on the `geometry` column of a GeoPandas dataframe"
    df = df.copy()
    df = df.to_crs(epsg=3857)  # web mercator

    if ax is None:
        _, ax = plt.subplots(figsize=(8,8))
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    df.plot(
        ax=ax,
        alpha=0.5, edgecolor='k',
        column=column, categorical=True,
        legend=True, legend_kwds={'loc': 'upper left'},
    )
    cx.add_basemap(ax, crs=df.crs, source=cx.providers.CartoDB.Positron)
    
    
def plot_shape(shape, ax=None):
    df = geopandas.GeoDataFrame({'geometry': [shape]}, crs='EPSG:4326')
    plot_df(df, ax=ax)
    
def plot_cells(cells, ax=None):
    shape = h3.cells_to_h3shape(cells)
    plot_shape(shape, ax=ax)

In [195]:
import folium
import h3
import pandas as pd
import numpy as np
import geopandas as gpd
import requests
import json
from folium.plugins import HeatMap
from shapely.geometry import Point, Polygon
import warnings
warnings.filterwarnings('ignore')

class H3NigeriaMapper:
    def __init__(self):
        self.map = None
        self.h3_resolution = 7  # Adjust based on your needs (6-9 typical for city-level)
        
    def create_base_map(self, center_lat=9.0820, center_lon=8.6753, zoom=6):
        """Create base map centered on Nigeria"""
        self.map = folium.Map(
            location=[center_lat, center_lon],
            zoom_start=zoom,
            tiles='OpenStreetMap'
        )
        return self.map
    
    def load_sample_data(self, n_points=1000):
        """Generate sample data points across Nigeria"""
        # Nigeria approximate bounds
        lat_min, lat_max = 4.0, 14.0
        lon_min, lon_max = 2.5, 15.0
        
        np.random.seed(42)
        data = {
            'latitude': np.random.uniform(lat_min, lat_max, n_points),
            'longitude': np.random.uniform(lon_min, lon_max, n_points),
            'value': np.random.randint(1, 100, n_points),  # Some metric to aggregate
            'category': np.random.choice(['A', 'B', 'C'], n_points)
        }
        return pd.DataFrame(data)
    
    def points_to_h3(self, df, lat_col='latitude', lon_col='longitude'):
        """Convert lat/lon points to H3 hexagons"""
        df['h3_index'] = df.apply(
            lambda row: h3.latlng_to_cell(
                row[lat_col], row[lon_col], self.h3_resolution
            ), axis=1
        )
        return df
    
    def aggregate_by_h3(self, df, value_col='value'):
        """Aggregate points by H3 hexagon"""
        h3_stats = df.groupby('h3_index').agg({
            value_col: ['count', 'sum', 'mean'],
            'category': lambda x: x.mode().iloc[0] if not x.empty else 'Unknown'
        }).round(2)
        
        # Flatten column names
        h3_stats.columns = ['point_count', 'total_value', 'avg_value', 'dominant_category']
        h3_stats = h3_stats.reset_index()
        
        # Get hexagon boundaries
        h3_stats['geometry'] = h3_stats['h3_index'].apply(
            lambda x: Polygon(h3.cell_to_boundary(x))#, geo_json=True
        )
        
        return gpd.GeoDataFrame(h3_stats, geometry='geometry')
    
    def add_h3_hexagons(self, h3_gdf, color_col='point_count'):
        """Add H3 hexagons to map with color coding"""
        # Normalize values for color mapping
        min_val = h3_gdf[color_col].min()
        max_val = h3_gdf[color_col].max()
        
        def get_color(value):
            # Color scale from light to dark blue
            normalized = (value - min_val) / (max_val - min_val) if max_val > min_val else 0
            opacity = 0.3 + (normalized * 0.7)
            return f'rgba(0, 100, 200, {opacity})'
        
        for idx, row in h3_gdf.iterrows():
            # Convert geometry to GeoJSON format
            coords = [[list(coord) for coord in row.geometry.exterior.coords]]
            
            popup_html = f"""
            <div style="font-family: Arial; font-size: 12px;">
                <b>H3 Index:</b> {row.h3_index}<br>
                <b>Points:</b> {row.point_count}<br>
                <b>Total Value:</b> {row.total_value}<br>
                <b>Avg Value:</b> {row.avg_value}<br>
                <b>Category:</b> {row.dominant_category}
            </div>
            """
            
            folium.Polygon(
                locations=coords,
                popup=folium.Popup(popup_html, max_width=300),
                color='blue',
                weight=1,
                fill=True,
                fillColor=get_color(row[color_col]),
                fillOpacity=0.7
            ).add_to(self.map)
    
    def load_nigeria_boundaries(self, admin_level='state'):
        """Load Nigerian administrative boundaries from GeoJSON files"""
        import os
        
        base_path = "../input/geojson"
        
        try:
            if admin_level == 'lga':
                file_path = os.path.join(base_path, "GRID3_NGA_-_Operational_LGA_Boundaries.geojson")
                with open(file_path, 'r', encoding='utf-8') as f:
                    return json.load(f)
            
            elif admin_level == 'ward':
                file_path = os.path.join(base_path, "Nigeria_-_Ward_Boundaries.geojson")
                with open(file_path, 'r', encoding='utf-8') as f:
                    return json.load(f)
            
            elif admin_level == 'state':
                # Extract state boundaries from LGA data by dissolving
                lga_gdf = gpd.read_file(os.path.join(base_path, "GRID3_NGA_-_Operational_LGA_Boundaries.geojson"))
                
                # Assuming there's a state column in the LGA data
                state_col = None
                for col in lga_gdf.columns:
                    if 'state' in col.lower() or 'admin1' in col.lower():
                        state_col = col
                        break
                
                if state_col:
                    # Dissolve LGA boundaries to create state boundaries
                    states_gdf = lga_gdf.dissolve(by=state_col).reset_index()
                    return json.loads(states_gdf.to_json())
                else:
                    print("Warning: Could not find state column in LGA data")
                    return self._create_nigeria_outline()
            
            else:
                return self._create_nigeria_outline()
                
        except FileNotFoundError as e:
            print(f"Error loading {admin_level} boundaries: {e}")
            return self._create_nigeria_outline()
        except Exception as e:
            print(f"Error processing {admin_level} boundaries: {e}")
            return self._create_nigeria_outline()
    
    def inspect_geojson_structure(self):
        """Inspect the structure of your GeoJSON files to understand the data"""
        import os
        
        base_path = "../input/geojson"
        files = [
            "GRID3_NGA_-_Operational_LGA_Boundaries.geojson",
            "Nigeria_-_Ward_Boundaries.geojson"
        ]
        
        for filename in files:
            file_path = os.path.join(base_path, filename)
            try:
                print(f"\n=== {filename} ===")
                gdf = gpd.read_file(file_path)
                print(f"Shape: {gdf.shape}")
                print(f"Columns: {list(gdf.columns)}")
                print(f"CRS: {gdf.crs}")
                print("\nFirst few rows of properties:")
                print(gdf.drop('geometry', axis=1).head(2))
                
                # Show unique values for potential grouping columns
                for col in gdf.columns:
                    if col != 'geometry' and gdf[col].dtype == 'object':
                        unique_count = gdf[col].nunique()
                        if unique_count < 50:  # Only show if not too many unique values
                            print(f"\nUnique values in '{col}' ({unique_count}): {sorted(gdf[col].unique())[:10]}")
                
            except Exception as e:
                print(f"Error reading {filename}: {e}")
        
        return None
    
    def add_admin_boundaries(self, admin_levels=['state', 'lga', 'ward']):
        """Add Nigerian administrative boundaries as overlays"""
        
        # Layer control
        boundary_layers = {}
        
        for level in admin_levels:
            # Create feature group for this admin level
            fg = folium.FeatureGroup(name=f'{level.upper()} Boundaries')
            
            # Load boundaries
            boundary_data = self.load_nigeria_boundaries(level)
            
            # Style configuration for different admin levels
            style_config = {
                'state': {'color': 'red', 'weight': 3, 'opacity': 0.8},
                'lga': {'color': 'orange', 'weight': 2, 'opacity': 0.7},
                'ward': {'color': 'yellow', 'weight': 1, 'opacity': 0.6}
            }
            
            # Function to create popup content
            def create_popup(feature):
                props = feature.get('properties', {})
                
                # Try to find name field (common variations)
                name_fields = ['name', 'Name', 'NAME', 'admin_name', 'lga_name', 'ward_name', 'ADM1_EN', 'ADM2_EN', 'ADM3_EN']
                name = 'Unknown'
                for field in name_fields:
                    if field in props:
                        name = props[field]
                        break
                
                # Create popup content with available properties
                popup_content = f"<b>{level.upper()}:</b> {name}<br>"
                
                # Add additional properties if available
                for key, value in props.items():
                    if key not in name_fields and len(str(value)) < 50:  # Avoid very long values
                        popup_content += f"<b>{key}:</b> {value}<br>"
                
                return popup_content
            
            # Add to feature group
            folium.GeoJson(
                boundary_data,
                style_function=lambda feature, style=style_config[level]: {
                    'fillColor': 'transparent',
                    'color': style['color'],
                    'weight': style['weight'],
                    'opacity': style['opacity'],
                    'fillOpacity': 0.1
                },
                popup=folium.Popup(
                    lambda feature: create_popup(feature),
                    max_width=300
                ),
                tooltip=folium.Tooltip(
                    lambda feature: f"{level.upper()}: {self._get_feature_name(feature)}"
                )
            ).add_to(fg)
            
            boundary_layers[level] = fg
        
        return boundary_layers
    
    def _get_feature_name(self, feature):
        """Extract name from feature properties"""
        props = feature.get('properties', {})
        name_fields = ['name', 'Name', 'NAME', 'admin_name', 'lga_name', 'ward_name', 'ADM1_EN', 'ADM2_EN', 'ADM3_EN']
        
        for field in name_fields:
            if field in props:
                return props[field]
        
        return 'Unknown'
    
    def create_full_map(self, df=None, save_path='nigeria_h3_map.html'):
        """Create complete map with all components"""
        
        # Create base map
        self.create_base_map()
        
        # Load or use provided data
        if df is None:
            df = self.load_sample_data(5000)  # 5K points for demo
        
        # Convert to H3 and aggregate
        df_h3 = self.points_to_h3(df)
        h3_aggregated = self.aggregate_by_h3(df_h3)
        
        # Add H3 hexagons
        self.add_h3_hexagons(h3_aggregated)
        
        # Add administrative boundaries
        boundary_layers = self.add_admin_boundaries()
        
        # Add boundary layers to map
        for layer_name, layer in boundary_layers.items():
            layer.add_to(self.map)
        
        # Add layer control
        folium.LayerControl().add_to(self.map)
        
        # Add legend
        legend_html = '''
        <div style="position: fixed; 
                    bottom: 50px; left: 50px; width: 200px; height: 120px; 
                    background-color: white; border:2px solid grey; z-index:9999; 
                    font-size:14px; padding: 10px">
        <h4>Legend</h4>
        <p><i class="fa fa-square" style="color:blue"></i> H3 Hexagons (Point Density)</p>
        <p><i class="fa fa-square" style="color:red"></i> State Boundaries</p>
        <p><i class="fa fa-square" style="color:orange"></i> LGA Boundaries</p>
        <p><i class="fa fa-square" style="color:yellow"></i> Ward Boundaries</p>
        </div>
        '''
        self.map.get_root().html.add_child(folium.Element(legend_html))
        
        # Save map
        self.map.save(save_path)
        print(f"Map saved to {save_path}")
        
        return self.map, h3_aggregated

# Example usage for clustering points WITHIN hexagons
def cluster_points_within_hexagons(df, h3_resolution=7):
    """
    Example of clustering points within H3 hexagons using DBSCAN
    This is more advanced and computationally intensive
    """
    from sklearn.cluster import DBSCAN
    
    # Convert to H3
    df['h3_index'] = df.apply(
        lambda row: h3.latlng_to_cell(row['latitude'], row['longitude'], h3_resolution), 
        axis=1
    )
    
    results = []
    
    # For each hexagon, cluster points within it
    for h3_idx, group in df.groupby('h3_index'):
        if len(group) < 2:  # Need at least 2 points to cluster
            continue
            
        # Prepare coordinates for clustering
        coords = group[['latitude', 'longitude']].values
        
        # Apply DBSCAN clustering
        clustering = DBSCAN(eps=0.01, min_samples=2).fit(coords)
        
        # Add cluster labels
        group = group.copy()
        group['cluster_id'] = clustering.labels_
        group['h3_cluster'] = group.apply(
            lambda row: f"{row['h3_index']}_{row['cluster_id']}", axis=1
        )
        
        results.append(group)
    
    return pd.concat(results, ignore_index=True)

    def _create_nigeria_outline(self):
        """Create a simple Nigeria country outline as fallback"""
        return {
            "type": "Feature",
            "geometry": {
                "type": "Polygon",
                "coordinates": [[[2.5, 4.0], [15.0, 4.0], [15.0, 14.0], [2.5, 14.0], [2.5, 4.0]]]
            },
            "properties": {"name": "Nigeria"}
        }



In [192]:
mapper = H3NigeriaMapper()
# mapper.inspect_geojson_structure()

In [197]:

# Usage example with inspection
if __name__ == "__main__":
    # Initialize mapper
    mapper = H3NigeriaMapper()
    
    # First, inspect your GeoJSON files to understand the structure
    print("Inspecting GeoJSON files...")
    # mapper.inspect_geojson_structure()
    
    print("\n" + "="*50)
    print("Creating map...")
    
    # Create the map
    nigeria_map, h3_data = mapper.create_full_map()
    
    # Print summary statistics
    print("H3 Aggregation Summary:")
    print(h3_data[['point_count', 'total_value', 'avg_value']].describe())
    
    

Inspecting GeoJSON files...

Creating map...
Error processing state boundaries: Object of type Timestamp is not JSON serializable


AttributeError: 'H3NigeriaMapper' object has no attribute '_create_nigeria_outline'